
# PCA Geometry: Rotate, then Rescale (Whiten)
This notebook demonstrates PCA as two consecutive steps:
1. **Rotation** into the eigenvector basis (uncorrelated components, different variances).
2. **Rescaling** (whitening) to unit variance.

At each step we **overlay eigenvectors** on the point cloud. All plots use `matplotlib` only, one chart per cell.


In [1]:

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
from scipy.linalg import eigh

%matplotlib inline

def center_rows(X):
    """Center each row (2xN)."""
    return X - X.mean(axis=1, keepdims=True)

def cov_pop(X):
    """Population covariance for 2xN samples."""
    N = X.shape[1]
    return (X @ X.T) / N

def corr_from_cov(C):
    d = np.sqrt(np.diag(C))
    if np.any(d==0): return 0.0
    return C[0,1]/(d[0]*d[1])

def draw_cloud_with_eigs(X, Cov, title="", lim=5.0, bins=101, scale=2.0, equal_axis=True):
    """2D histogram + overlay eigenvectors scaled by sqrt(eigenvalues)."""
    vals, vecs = eigh(Cov)  # ascending λ1 <= λ2
    plt.figure(figsize=(5,5))
    plt.hist2d(X[0], X[1], bins=bins, range=[[-lim, lim], [-lim, lim]])
    # Overlay eigenvectors from origin
    for k in range(2):
        v = vecs[:,k] * np.sqrt(max(vals[k], 0.0)) * scale
        plt.plot([0, v[0]], [0, v[1]], '.-', linewidth=2)
    plt.xlabel('dim 1'); plt.ylabel('dim 2')
    ttl = title
    ttl += f"\ncorrelation = {corr_from_cov(Cov):.3f}"
    plt.title(ttl)
    if equal_axis: plt.axis('equal')
    plt.xlim(-lim, lim); plt.ylim(-lim, lim)
    plt.show()

def simulate_linear(noise=1.0, N=int(1e5), seed=None):
    """x ~ N(0,1), y = x + noise*eps; return centered X (2xN)."""
    rng = np.random.default_rng(seed)
    x = rng.normal(size=N)
    y = x + noise*rng.normal(size=N)
    X = np.vstack([x, y])
    return center_rows(X)

def pca_rotation_then_whitening(noise=1.0, N=int(1e5), lim=5.0, bins=101, show_equal_axis=True):
    # Raw data
    X = simulate_linear(noise=noise, N=N)
    Cov = cov_pop(X)

    # Step 1: Rotation (PCA): Y = V^T X
    evals, evecs = eigh(Cov)     # Cov = V Λ V^T; evals ascending
    Y = evecs.T @ X              # rotated data
    CovY = cov_pop(Y)            # should be diag(evals)

    # Step 2: Rescaling (Whitening): Z = Λ^{-1/2} Y
    eps = 1e-12
    inv_sqrt = np.diag(1.0/np.sqrt(np.maximum(evals, eps)))
    Z = inv_sqrt @ Y
    CovZ = cov_pop(Z)            # ~ I

    # Plots are generated by separate functions below
    return (X, Cov, Y, CovY, Z, CovZ, evals, evecs)



## Interactive controls
Use the sliders to change noise, sample size, histogram bins, and axes limits.  
Run the three cells below (Raw → Rotated → Whitened) to see each step with eigenvectors overlaid.


In [2]:

def view_raw(noise=1.0, N=100000, bins=101, lim=5.0, equal_axis=True):
    X = simulate_linear(noise=noise, N=N)
    Cov = cov_pop(X)
    draw_cloud_with_eigs(X, Cov, title="Raw data (correlated)", lim=lim, bins=bins, equal_axis=equal_axis)

interact(view_raw,
         noise=FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description='noise σ'),
         N=IntSlider(value=100000, min=2000, max=200000, step=5000, description='N'),
         bins=IntSlider(value=101, min=31, max=201, step=5, description='bins'),
         lim=FloatSlider(value=5.0, min=2.0, max=10.0, step=0.5, description='axis lim'),
         equal_axis=Checkbox(value=True, description='equal axis'));


interactive(children=(FloatSlider(value=1.0, description='noise σ', max=3.0), IntSlider(value=100000, descript…

In [3]:

def view_rotated(noise=1.0, N=100000, bins=101, lim=5.0, equal_axis=True):
    X = simulate_linear(noise=noise, N=N)
    Cov = cov_pop(X)
    evals, evecs = eigh(Cov)
    Y = evecs.T @ X
    # In rotated space, eigenvectors are canonical axes; use CovY for overlay
    CovY = np.diag(evals)
    draw_cloud_with_eigs(Y, CovY, title="After PCA rotation (uncorrelated; variances = eigenvalues)", 
                         lim=lim, bins=bins, equal_axis=equal_axis)

interact(view_rotated,
         noise=FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description='noise σ'),
         N=IntSlider(value=100000, min=2000, max=200000, step=5000, description='N'),
         bins=IntSlider(value=101, min=31, max=201, step=5, description='bins'),
         lim=FloatSlider(value=5.0, min=2.0, max=10.0, step=0.5, description='axis lim'),
         equal_axis=Checkbox(value=True, description='equal axis'));


interactive(children=(FloatSlider(value=1.0, description='noise σ', max=3.0), IntSlider(value=100000, descript…

In [5]:

def view_whitened(noise=1.0, N=100000, bins=101, lim=5.0, equal_axis=True):
    X = simulate_linear(noise=noise, N=N)
    Cov = cov_pop(X)
    evals, evecs = eigh(Cov)
    Y = evecs.T @ X
    eps = 1e-12
    inv_sqrt = np.diag(1.0/np.sqrt(np.maximum(evals, eps)))
    Z = inv_sqrt @ Y
    CovZ = cov_pop(Z)  # ~ I
    draw_cloud_with_eigs(Z, CovZ, title="After whitening (uncorrelated; unit variance)", 
                         lim=lim, bins=bins, equal_axis=equal_axis)

interact(view_whitened,
         noise=FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description='noise σ'),
         N=IntSlider(value=100000, min=2000, max=200000, step=5000, description='N'),
         bins=IntSlider(value=101, min=31, max=201, step=5, description='bins'),
         lim=FloatSlider(value=5.0, min=2.0, max=10.0, step=0.5, description='axis lim'),
         equal_axis=Checkbox(value=True, description='equal axis'));


interactive(children=(FloatSlider(value=1.0, description='noise σ', max=3.0), IntSlider(value=100000, descript…


## Verify covariances
Run this cell to print the covariance matrices at each step.


In [ ]:

def print_covariances(noise=1.0, N=100000):
    X = simulate_linear(noise=noise, N=N)
    Cov = cov_pop(X)
    evals, evecs = eigh(Cov)
    Y = evecs.T @ X
    CovY = cov_pop(Y)
    eps = 1e-12
    inv_sqrt = np.diag(1.0/np.sqrt(np.maximum(evals, eps)))
    Z = inv_sqrt @ Y
    CovZ = cov_pop(Z)
    print("Cov (raw):\n", Cov)
    print("\nCov after rotation (diag):\n", CovY)
    print("\nCov after whitening (≈ I):\n", CovZ)

from ipywidgets import interact
interact(print_covariances,
         noise=FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description='noise σ'),
         N=IntSlider(value=100000, min=2000, max=200000, step=5000, description='N'));
